In [1]:
print("Hello world")

Hello world


In [2]:
from pyspark import SparkContext
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName("test-fixed-width").getOrCreate()

26/03/12 11:05:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
import logging

# Reduce logging verbosity
spark.sparkContext.setLogLevel("ERROR")  # Options: ALL, DEBUG, INFO, WARN, ERROR, FATAL

In [11]:
# Below are some tests to check if the fixedwidthloader actually is working like the csv loader that exists natively within databricks

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    #StructField("id", IntegerType(), True),
    StructField("id", StringType(), True)
])

df = spark.read.format("fixedwidth-custom-scala") \
    .option("field_lengths", "0:5,6:14") \
    .schema(schema) \
     .load("../data/test_data/valid1.txt")\
    #.schema(schema) \
    #.load("data/test_data/valid1.txt")

df.show()

+-----+----+
| name|  id|
+-----+----+
|Alice|0001|
|  Bob|0002|
|Carol|0003|
+-----+----+



In [5]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("id", IntegerType(), True),
])

df = spark.read.format("fixedwidth-custom-scala") \
     .option("field_lengths", "0:5,6:10") \
     .option("rescuedDataColumn", "_rescued_data") \
    .schema(schema) \
     .load("../data/test_data/valid1.txt")


df.show()

+-----+---+-------------+
| name| id|_rescued_data|
+-----+---+-------------+
|Alice|  1|         NULL|
|  Bob|  2|         NULL|
|Carol|  3|         NULL|
+-----+---+-------------+



In [6]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("id", IntegerType(), True),
    StructField("_rescued_data", StringType(), True),
    #StructField("_corrupt_record", StringType(), nullable=True),
])

#df = spark.read.format("fixedwidth-custom-scala") \
#     .option("field_lengths", "0:5,6:10") \
#     .option("rescuedDataColumn", "_rescued_data") \
#    .schema(schema) \
#     .load("../data/test_data/invalid1.txt")


df = spark.read.format("fixedwidth-custom-scala") \
     .option("field_lengths", "0:5,6:10") \
    .schema(schema) \
     .load("../data/test_data/invalid1.txt")



df.show()

+-----+----+-------------+
| name|  id|_rescued_data|
+-----+----+-------------+
|Alice|   1|         NULL|
|  Bob|   2|         NULL|
|Carol|NULL|         NULL|
+-----+----+-------------+



In [7]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("id", IntegerType(), True),
])

df = spark.read.format("fixedwidth-custom-scala") \
     .option("field_lengths", "0:5,6:10") \
    .option("mode", "PERMISSIVE") \
     .option("rescuedDataColumn", "_rescued_data") \
    .schema(schema) \
     .load("../data/test_data/invalid1.txt")


df.show()

+-----+----+--------------------+
| name|  id|       _rescued_data|
+-----+----+--------------------+
|Alice|   1|                NULL|
|  Bob|   2|                NULL|
|Carol|NULL|{"id":"000A","_fi...|
+-----+----+--------------------+



In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("id", IntegerType(), True),
     StructField("id", IntegerType(), True),
])


df = spark.read.format("fixedwidth-custom-scala") \
     .option("field_lengths", "0:5,6:10") \
     .option("mode", "PERMISSIVE") \
     .schema(schema) \
     .load("../data/test_data/invalid1.txt")


df.show()

+-----+----+----+
| name|  id|  id|
+-----+----+----+
|Alice|   1|NULL|
|  Bob|   2|NULL|
|Carol|NULL|NULL|
+-----+----+----+



In [9]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("id", IntegerType(), True),
     StructField("id", IntegerType(), True),
    StructField("_corrupt_record", StringType(), nullable=True)
])


df = spark.read.format("fixedwidth-custom-scala") \
     .option("field_lengths", "0:5,6:10") \
     .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record")\
     .schema(schema) \
     .load("../data/test_data/invalid1.txt")


df.show()

+-----+----+----+---------------+
| name|  id|  id|_corrupt_record|
+-----+----+----+---------------+
|Alice|   1|NULL|           NULL|
|  Bob|   2|NULL|           NULL|
|Carol|NULL|NULL|     Carol,000A|
+-----+----+----+---------------+



In [10]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("id", IntegerType(), True),
     StructField("id", IntegerType(), True),
    StructField("_corrupt_record", StringType(), nullable=True)
])


df = spark.read.format("fixedwidth-custom-scala") \
     .option("field_lengths", "0:5,6:10") \
     .option("mode", "PERMISSIVE") \
     .schema(schema) \
     .load("../data/test_data/invalid1.txt")


df.show()

+-----+----+----+---------------+
| name|  id|  id|_corrupt_record|
+-----+----+----+---------------+
|Alice|   1|NULL|           NULL|
|  Bob|   2|NULL|           NULL|
|Carol|NULL|NULL|     Carol,000A|
+-----+----+----+---------------+



In [11]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("id", IntegerType(), True),
     StructField("id", IntegerType(), True),
    StructField("_corrupt_record", StringType(), nullable=True)
])


df = spark.read.format("fixedwidth-custom-scala") \
     .option("field_lengths", "0:5,6:10") \
     .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record")\
    .option("rescuedDataColumn", "_rescued_data")\
     .schema(schema) \
     .load("../data/test_data/invalid1.txt")


df.show()

+-----+----+----+---------------+--------------------+
| name|  id|  id|_corrupt_record|       _rescued_data|
+-----+----+----+---------------+--------------------+
|Alice|   1|NULL|           NULL|                NULL|
|  Bob|   2|NULL|           NULL|                NULL|
|Carol|NULL|NULL|           NULL|{"id":"000A","_fi...|
+-----+----+----+---------------+--------------------+



In [12]:
# Test: rescued data _file_path controlled by Spark SQL config
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("id", IntegerType(), True),
])

# Default: _file_path should be present
df1 = spark.read.format("fixedwidth-custom-scala") \
    .option("field_lengths", "0:5,5:10") \
    .option("rescuedDataColumn", "_rescued_data") \
    .schema(schema) \
    .load("../data/test_data/invalid1.txt")
print("=== Default (file_path included) ===")
df1.select("name", "id", "_rescued_data").show(truncate=False)

# Disable _file_path
spark.conf.set("spark.databricks.sql.rescuedDataColumn.filePath.enabled", "false")
df2 = spark.read.format("fixedwidth-custom-scala") \
    .option("field_lengths", "0:5,5:10") \
    .option("rescuedDataColumn", "_rescued_data") \
    .schema(schema) \
    .load("../data/test_data/invalid1.txt")
print("=== file_path disabled ===")
df2.select("name", "id", "_rescued_data").show(truncate=False)

# Reset
spark.conf.unset("spark.databricks.sql.rescuedDataColumn.filePath.enabled")

=== Default (file_path included) ===
+-----+----+--------------------------------------------------------------------------------------------------------------------+
|name |id  |_rescued_data                                                                                                       |
+-----+----+--------------------------------------------------------------------------------------------------------------------+
|Alice|1   |NULL                                                                                                                |
|Bob  |2   |NULL                                                                                                                |
|Carol|NULL|{"id":"0000A","_file_path":"file:/workspaces/spark-databricks-fixedwidth-datasource-v2/data/test_data/invalid1.txt"}|
+-----+----+--------------------------------------------------------------------------------------------------------------------+

=== file_path disabled ===
+-----+----+-------------